In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, accuracy_score, cohen_kappa_score, confusion_matrix
import os
import warnings
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# COLAB SETUP — delete this cell and the one below if running locally
Run the cell below only in Colab. It installs missing packages, uploads the two
required CSVs, and sets Colab-specific paths.

In [ ]:
!pip install -q transformers accelerate torch

from google.colab import files
print("Upload malaysian_sentiment_labeled.csv")
uploaded1 = files.upload()
print("Upload gold_labeled.csv")
uploaded2 = files.upload()

TRAIN_PATH = list(uploaded1.keys())[0]
TEST_PATH = list(uploaded2.keys())[0]
SAVE_DIR = '/content/xlmr_finetuned'
RESULTS_FILE = '/content/colab_model_results.csv'

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'[*_]{1,2}([^*_]+)[*_]{1,2}', r'\1', text)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'&gt;|&amp;|&lt;|&nbsp;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['cleaned'] = train_df['body'].apply(preprocess)
test_df['cleaned'] = test_df['body'].apply(preprocess)
train_df = train_df[train_df['cleaned'].str.split().apply(len) >= 3].reset_index(drop=True)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label = {v: k for k, v in label2id.items()}
train_df['label'] = train_df['sentiment'].map(label2id)
test_df['label'] = test_df['human_label'].map(label2id)

print(f'Train: {len(train_df)}  Test: {len(test_df)}')
print(train_df['sentiment'].value_counts())

In [ ]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
).to(device)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                              max_length=self.max_len, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = SentimentDataset(train_df['cleaned'].tolist(), train_df['label'].tolist(), tokenizer)
test_dataset = SentimentDataset(test_df['cleaned'].tolist(), test_df['label'].tolist(), tokenizer)
print('Datasets ready')

In [ ]:
class_counts = train_df['label'].value_counts().sort_index()
raw_weights = [len(train_df) / (3 * c) for c in class_counts]
class_weights = torch.tensor([min(w, 5.0) for w in raw_weights], dtype=torch.float).to(device)
print(f'Class weights: {class_weights}')

from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='../models/transformers/xlmr_finetuned_checkpoints',
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=50,
    save_strategy='no',
    eval_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

In [ ]:
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
pred_labels = [id2label[p] for p in preds]
y_test = test_df['human_label'].tolist()

acc = accuracy_score(y_test, pred_labels)
kappa = cohen_kappa_score(y_test, pred_labels)
report = classification_report(y_test, pred_labels, target_names=['negative','neutral','positive'], output_dict=True)

print(f'XLM-R Fine-Tuned')
print(f'Accuracy: {acc*100:.1f}%   Kappa: {kappa:.3f}')
print(classification_report(y_test, pred_labels, target_names=['negative','neutral','positive']))

labels = ['negative','neutral','positive']
cm = confusion_matrix(y_test, pred_labels, labels=labels)
print(f"{'':>10}" + "".join(f"{l:>10}" for l in labels))
for i, l in enumerate(labels):
    print(f"{l:>10}" + "".join(f"{cm[i][j]:>10}" for j in range(len(labels))))

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Saved model and tokenizer to {SAVE_DIR}')

new_row = pd.DataFrame([{
    'model': 'XLM-R Fine-Tuned',
    'status': 'transformer_finetuned',
    'accuracy': acc,
    'kappa': kappa,
    'neg_f1': report['negative']['f1-score'],
    'neu_f1': report['neutral']['f1-score'],
    'pos_f1': report['positive']['f1-score'],
}])
existing = pd.read_csv(RESULTS_FILE)
existing = existing[existing['model'] != 'XLM-R Fine-Tuned']
combined = pd.concat([existing, new_row], ignore_index=True)
combined.to_csv(RESULTS_FILE, index=False)
print(combined.to_string(index=False))

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/xlmr_finetuned', 'zip', SAVE_DIR)
files.download('/content/xlmr_finetuned.zip')
files.download(RESULTS_FILE)